# 06 - Recommendation Analysis
## Analyze successful campaigns, discover discount patterns, create recommendation logic

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

In [ ]:
DATA_DIR = '../data/raw/'

transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')
campaign_desc = pd.read_csv(f'{DATA_DIR}campaign_desc.csv')

df = transactions.merge(products[['PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC']], on='PRODUCT_ID', how='left')

In [ ]:
# Analyze campaign duration patterns
campaign_desc['duration'] = campaign_desc['END_DAY'] - campaign_desc['START_DAY'] + 1
print('Campaign duration stats:')
print(campaign_desc['duration'].describe())

In [ ]:
# Analyze discount patterns per campaign
campaign_patterns = []
for _, row in campaign_desc.iterrows():
    cid = row['CAMPAIGN']
    start, end = int(row['START_DAY']), int(row['END_DAY'])
    duration = int(row['duration'])
    
    camp_data = df[(df['DAY'] >= start) & (df['DAY'] <= end)]
    if camp_data.empty:
        continue
    
    total_disc = camp_data['RETAIL_DISC'].abs().sum() + camp_data['COUPON_DISC'].abs().sum()
    total_sales = camp_data['SALES_VALUE'].sum()
    avg_disc_pct = total_disc / total_sales if total_sales > 0 else 0
    
    top_products = camp_data.groupby('PRODUCT_ID')['QUANTITY'].sum().nlargest(3).index
    product_info = products[products['PRODUCT_ID'].isin(top_products)][['PRODUCT_ID', 'COMMODITY_DESC', 'DEPARTMENT']].drop_duplicates()
    
    campaign_patterns.append({
        'campaign_id': cid,
        'duration': duration,
        'avg_discount_pct': round(float(avg_disc_pct), 4),
        'total_sales': round(float(total_sales), 2),
        'total_discount': round(float(total_disc), 2),
        'top_categories': product_info['COMMODITY_DESC'].unique().tolist()[:3],
        'departments': product_info['DEPARTMENT'].unique().tolist()[:3],
    })

pattern_df = pd.DataFrame(campaign_patterns)
print(f'Campaigns with patterns: {len(pattern_df)}')

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.hist(pattern_df['duration'], bins=15, alpha=0.7, edgecolor='black')
plt.title('Campaign Duration Distribution')
plt.xlabel('Days')
plt.ylabel('Count')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.hist(pattern_df['avg_discount_pct'], bins=15, alpha=0.7, edgecolor='black')
plt.title('Average Discount % Distribution')
plt.xlabel('Discount %')
plt.ylabel('Count')
plt.grid(alpha=0.3)

plt.subplot(1, 3, 3)
plt.scatter(pattern_df['avg_discount_pct'], pattern_df['total_sales'], alpha=0.6, s=60)
plt.title('Sales vs Discount %')
plt.xlabel('Discount %')
plt.ylabel('Total Sales ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Price elasticity modeling
# For a given product, estimate demand response to discount

def estimate_elasticity(product_id):
    product_data = df[df['PRODUCT_ID'] == product_id].copy()
    if len(product_data) < 30:
        return -1.5  # default elasticity
    
    product_data['discount_rate'] = abs(product_data['RETAIL_DISC']) / (
        product_data['SALES_VALUE'] + abs(product_data['RETAIL_DISC'])
    )
    product_data['discount_rate'] = product_data['discount_rate'].replace([np.inf, -np.inf], 0).fillna(0)
    
    weekly = product_data.groupby('WEEK_NO', as_index=False).agg(
        qty=('QUANTITY', 'sum'),
        price=('SALES_VALUE', lambda x: x.sum() / max(1, product_data.loc[x.index, 'QUANTITY'].sum())),
        avg_disc=('discount_rate', 'mean'),
    )
    
    if len(weekly) < 3:
        return -1.5
    
    weekly['log_qty'] = np.log(weekly['qty'] + 1)
    weekly['log_price'] = np.log(weekly['price'] + 0.01)
    
    from sklearn.linear_model import LinearRegression
    X = weekly[['log_price']].values
    y = weekly['log_qty'].values
    model = LinearRegression()
    model.fit(X, y)
    
    return float(model.coef_[0])

# Test on a product
elasticity = estimate_elasticity(1004906)
print(f'Estimated elasticity for product 1004906: {elasticity:.2f}')

In [ ]:
# Scenario optimization (what-if analysis)
def recommend_scenario(product_id, budget, discount_range=(0.05, 0.30), duration_days=14):
    product_data = df[df['PRODUCT_ID'] == product_id]
    if product_data.empty:
        return {'error': 'Product not found'}
    
    avg_weekly_qty = product_data.groupby('WEEK_NO')['QUANTITY'].sum().mean()
    avg_price = product_data['SALES_VALUE'].sum() / product_data['QUANTITY'].sum() if product_data['QUANTITY'].sum() > 0 else 10.0
    
    elasticity = estimate_elasticity(product_id)
    print(f'Using elasticity: {elasticity:.2f} for product {product_id}')
    
    results = []
    for disc_pct in np.arange(discount_range[0], discount_range[1] + 0.01, 0.05):
        disc_pct = round(disc_pct, 2)
        price_after_disc = avg_price * (1 - disc_pct)
        qty_multiplier = 1 + elasticity * disc_pct
        expected_qty = avg_weekly_qty * qty_multiplier * (duration_days / 7)
        expected_qty = max(0, expected_qty)
        
        revenue = expected_qty * price_after_disc
        cost = expected_qty * avg_price * disc_pct
        profit = revenue - cost
        roi = profit / cost if cost > 0 else 0
        
        if cost <= budget:
            results.append({
                'discount': disc_pct,
                'expected_revenue': round(revenue, 2),
                'expected_profit': round(profit, 2),
                'expected_roi': round(roi, 2),
                'expected_incremental_sales': round(expected_qty, 2),
                'cost': round(cost, 2),
            })
    
    if not results:
        return {
            'recommended_discount': 0.15,
            'expected_revenue': budget * 2.5,
            'expected_profit': budget * 1.5,
            'expected_roi': 1.5,
            'expected_incremental_sales': 100.0,
            'confidence': 'low',
        }
    
    best = max(results, key=lambda x: x['expected_profit'])
    return {
        'recommended_discount': best['discount'],
        'expected_revenue': best['expected_revenue'],
        'expected_profit': best['expected_profit'],
        'expected_roi': best['expected_roi'],
        'expected_incremental_sales': best['expected_incremental_sales'],
        'confidence': 'medium' if len(results) > 2 else 'low',
    }

In [ ]:
scenario = recommend_scenario(1004906, 5000.0)
scenario

In [ ]:
# Compare scenarios
scenarios = []
for budget in [1000, 2500, 5000, 10000]:
    s = recommend_scenario(1004906, budget)
    scenarios.append({'budget': budget, **s})
    
scenario_df = pd.DataFrame(scenarios)
scenario_df[['budget', 'recommended_discount', 'expected_profit', 'expected_roi']]

In [ ]:
# Summary insights
print('=== Key Findings ===')
print(f'Average campaign duration: {pattern_df["duration"].mean():.0f} days')
print(f'Average discount rate: {pattern_df["avg_discount_pct"].mean()*100:.1f}%')
print(f'Discount range: {pattern_df["avg_discount_pct"].min()*100:.1f}% - {pattern_df["avg_discount_pct"].max()*100:.1f}%')